## 1. Opérations tensorielles typiques

On couvre : création, indexing, slicing, broadcasting, reshape et permute.

In [ ]:
import torch

print('Version PyTorch :', torch.__version__)


In [ ]:
x = torch.tensor([1., 2., 3.])
print('1.', x)

zeros = torch.zeros(2, 3)
print('2.\n', zeros)

random = torch.randn(2, 3)
print('3.\n', random)

print('4.', x[1])

print('5.', x[0:2])

a = torch.tensor([[1., 2.], [3., 4.]])
print('6.\n', a + 10)

b = torch.tensor([[1.], [2.]])
c = torch.tensor([10., 20., 30.])
print('7.\n', b + c)

r = torch.arange(12)
r2 = r.reshape(3, 4)
print('8.\n', r2)

p = torch.randn(2, 3, 4)
p2 = p.permute(2, 0, 1)
print('9. formes :', p.shape, '->', p2.shape)

m1 = torch.tensor([[1., 2.], [3., 4.]])
m2 = torch.tensor([[5., 6.], [7., 8.]])
print('10.\n', m1 @ m2)


### À retenir
- `shape` décrit les dimensions du tensor.
- `reshape()` change la forme.
- `permute()` réordonne les dimensions.
- Le broadcasting permet d'effectuer des opérations entre tensors de formes compatibles.

## 2. Autograd — 3 fonctions et vérification à la main

Pour obtenir automatiquement une dérivée, on crée un tensor avec `requires_grad=True`, puis on appelle `backward()`.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
f = x**2 + 3*x + 1
f.backward()

print('Gradient PyTorch :', x.grad.item())
print('Vérification manuelle : 2x + 3 =', 2*2 + 3)


**Vérification :**

f(x) = x² + 3x + 1

f'(x) = 2x + 3

Pour x = 2 : f'(2) = 7. PyTorch doit donc donner 7.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
f = x**3 + 2*x**2 - 5*x + 4
f.backward()

print('Gradient PyTorch :', x.grad.item())
    
print('Vérification manuelle : 3x^2 + 4x - 5 =', 3*(2**2) + 4*2 - 5)


**Vérification :**

f(x) = x³ + 2x² − 5x + 4

f'(x) = 3x² + 4x − 5

Pour x = 2 : 12 + 8 − 5 = **15**.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
f = (x**2 + 1)**3
f.backward()

print('Gradient PyTorch :', x.grad.item())
print('Vérification manuelle : 6x(x^2+1)^2 =', 6*2*(2**2 + 1)**2)


**Vérification :**

f(x) = (x² + 1)³

Avec la règle de la chaîne :

f'(x) = 3(x² + 1)² × 2x = 6x(x² + 1)²

Pour x = 2 : 6 × 2 × 25 = **300**.

Les trois gradients doivent donc être respectivement **7, 15 et 300**.

## 3. Régression linéaire complète avec SGD

On veut apprendre la relation :

**y = 3x + 2**

Le modèle doit donc apprendre un poids proche de `3` et un biais proche de `2`.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

X = torch.arange(0, 10, dtype=torch.float32).reshape(-1, 1)
y = 3 * X + 2

print('X :\n', X)
print('y :\n', y)


In [ ]:
model = nn.Linear(1, 1)

loss_fn = nn.MSELoss()

optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

print(model)


In [ ]:
for epoch in range(1000):
    y_pred = model(X)

    loss = loss_fn(y_pred, y)

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    if (epoch + 1) % 100 == 0:
        print(f'Epoch {epoch+1:4d} | loss = {loss.item():.6f}')


In [ ]:
weight = model.weight.item()
bias = model.bias.item()

print(f'Poids appris : {weight:.4f}')
print(f'Biais appris : {bias:.4f}')

x_new = torch.tensor([[10.0]])
prediction = model(x_new)

print(f'Pour x = 10, prédiction = {prediction.item():.4f}')
print('Valeur attendue : 32.0')


## 4. Comprendre la boucle d'entraînement

La séquence essentielle est toujours :

1. `model(X)` → prédiction
2. `loss_fn(...)` → mesure de l'erreur
3. `optimizer.zero_grad()` → efface les anciens gradients
4. `loss.backward()` → calcule les gradients
5. `optimizer.step()` → modifie les paramètres

C'est le mécanisme fondamental de l'apprentissage par descente de gradient dans PyTorch.

## Conclusion

Les notions maîtrisées dans ce notebook :

- créer et manipuler des tensors ;
- utiliser indexing, broadcasting, reshape et permute ;
- calculer des gradients avec autograd ;
- définir une fonction de perte ;
- utiliser `SGD` pour apprendre les paramètres ;
- comprendre la boucle `zero_grad → backward → step`.